In [ ]:
import parametrization, mesh, mesh_utilities, numpy as np
target_surface = mesh_utilities.subdivide_loop(mesh.Mesh("../examples/squidward_remesh.obj"), 1)
rparam = parametrization.RegularizedParametrizerSVD(target_surface, np.loadtxt('results/squidward/param_laptop.txt'))

In [ ]:
import visualization, wall_width_formulas as wwf
nsubdiv=3
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(3, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
upsampledStretches = np.clip(upsampledStretches, alphaMin, alphaMax)

In [ ]:
import pickle
sdfVertices, sdfTris, sdf = pickle.load(open('results/squidward/stripe_field_laptop.pkl', 'rb'))

Note: if `targetEdgeSpacing` is set too low relative to `triArea`, `triangle`
will insert new boundary points that fall in the interior of the flattened
target surface (in strictly convex regions) when refining the triangulation.

If the parametrization is subsequently used to lift the boundary points to 3D,
these lifted points will not lie on the target surface's boundary. E.g., they
may lift off the ground plane even if all boundary vertices of the target
surface lie on the ground plane.

In [ ]:
from sheet_meshing import generateSheetMesh
s, iwv, iwbv = generateSheetMesh(sdfVertices, sdfTris, sdf, triArea=0.05, permitWallInteriorVertices=False, targetEdgeSpacing=0.25)

In [ ]:
from tri_mesh_viewer import TriMeshViewer, RawMesh
sv = TriMeshViewer(s)
sv.showWireframe()
sv.show()

In [ ]:
import inflation, numpy as np, mesh_utilities
isheet = inflation.InflatableSheet(s, iwbv)
bv = isheet.mesh().boundaryVertices()
bdryVars = [isheet.varIdx(0, i, c) for i in bv for c in range(3)]

# uv = np.loadtxt('results/squidward/param_laptop_resave.txt')
uv = rparam.uv()
paramSampler = mesh_utilities.SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surface.triangles())
liftedSheetPositions = paramSampler.sample(s.vertices(), target_surface.vertices())

In [ ]:
uv_view.showWireframe(True)

In [ ]:
isheet.setIdentityDeformation()

In [ ]:
isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
import parametrization
harmonicPositions = parametrization.harmonic(isheet.mesh(), liftedSheetPositions[bv])
harmonicPositions += 0.15 * (liftedSheetPositions - harmonicPositions)

In [ ]:
isheet.setUninflatedDeformation(harmonicPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
import visualization
eqmesh = visualization.EquilibriumMesh(isheet)

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(eqmesh, width=1024, height=640)
#viewer.showWireframe()
viewer.setCameraParams(((2.855530256940937, 1.002991214638454, -1.1322350499425347),
 (-0.3303947111142514, -0.32128434449313503, -0.8874771573687668),
 (0.04746981475129009, 0.1255260177204086, 0.23082442618238475)))
viewer.show()

In [ ]:
# Target attracted sheet, allow closest points to slide again... debug equilibrium solver convergence issues / Hessian innacuracy
# Is the closest point projection imperfect?

In [ ]:
viewer.update(preserveExisting=True, mesh=target_surface)

In [ ]:
viewer.update()

In [ ]:
viewer.showWireframe(not viewer.shouldShowWireframe)

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-6
opts.niter = iterations_per_output

In [ ]:
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, target_surface)
# targetAttractedSheet.targetSurfaceFitter().closestSurfPts = np.loadtxt('results/squidward/debug/closestSurfPts.txt.gz')
targetAttractedSheet.fittingWeight = 1e-2

In [ ]:
import time
inflation.benchmark_reset()
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
niter = 5000
iterations_per_output = 10
opts.niter = iterations_per_output
# isheet.pressure = 1.0
isheet.pressure = 0.25
fixedVars = bdryVars
# fixedVars = isheet.rigidMotionPinVars
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(targetAttractedSheet, fixedVars, opts)
    # cr = inflation.inflation_newton(isheet, fixedVars, opts)
    # isheet.writeDebugMesh(f'results/squidward/hires_inflation/step_{step}.msh')
    # isheet.writeDebugMesh(f'results/squidward/inflation/step_{step}.msh')
    viewer.update()
    if cr.numIters() < iterations_per_output: break
inflation.benchmark_report()